# Stochastic Differential Equations (concept 33-sde)

An SDE is
$$dX_t = b(X_t, t)\,dt + \sigma(X_t, t)\,dW_t,$$
drift $b$ plus Brownian noise $\sigma\,dW_t$. Here we play with the
**Ornstein--Uhlenbeck** process
$$dX_t = -\theta X_t\,dt + \sigma\,dW_t,$$
the prototypical mean-reverting diffusion, and verify its stationary
variance $\sigma^2/(2\theta)$ via Euler--Maruyama.

Stdlib only --- no `numpy` required.

## 1. Euler--Maruyama integrator
Discretization: $X_{n+1} = X_n + b(X_n, t_n)\,\Delta t + \sigma(X_n, t_n)\sqrt{\Delta t}\,Z_n,\ Z_n\sim\mathcal N(0,1)$.

In [ ]:
import math, random

def euler_maruyama(b, sigma, x0, T, n_steps, seed=0):
    """Generic 1D Euler-Maruyama for dX = b(x,t) dt + sigma(x,t) dW."""
    rng = random.Random(seed)
    dt = T / n_steps
    sdt = math.sqrt(dt)
    x = x0
    ts = [0.0]
    xs = [x]
    for n in range(n_steps):
        t = n * dt
        z = rng.gauss(0.0, 1.0)
        x = x + b(x, t) * dt + sigma(x, t) * sdt * z
        ts.append((n + 1) * dt)
        xs.append(x)
    return ts, xs


## 2. Three Ornstein--Uhlenbeck sample paths
With $\theta=1$, $\sigma=0.5$, $X_0=2$ on $[0, 5]$, the deterministic
drift drags the path toward $0$; the noise jiggles it.

In [ ]:
theta, sigma_val, x0, T, n_steps = 1.0, 0.5, 2.0, 5.0, 1000
b = lambda x, t: -theta * x
s = lambda x, t: sigma_val

paths = []
for seed in (0, 1, 2):
    ts, xs = euler_maruyama(b, s, x0, T, n_steps, seed=seed)
    paths.append(xs)

# Print every 200-th step for a compact preview.
stride = n_steps // 5
print('t       ' + '  '.join('path{}'.format(i+1).rjust(8) for i in range(3)))
for k in range(0, n_steps + 1, stride):
    row = '{:5.2f}   '.format(k * T / n_steps)
    row += '  '.join('{:+8.4f}'.format(p[k]) for p in paths)
    print(row)


## 3. Stationary variance check
The OU process has $X_t\sim\mathcal N(X_0 e^{-\theta t},\,\tfrac{\sigma^2}{2\theta}(1-e^{-2\theta t}))$,
so $\mathrm{Var}(X_\infty) = \sigma^2/(2\theta) = 0.125$ for our parameters.
We run many short paths to a large $T$ and average $X_T^2$.

In [ ]:
T_long, n_long, n_paths = 20.0, 4000, 5000
s1 = 0.0
s2 = 0.0
for k in range(n_paths):
    _, xs = euler_maruyama(b, s, x0, T_long, n_long, seed=1000 + k)
    x_end = xs[-1]
    s1 += x_end
    s2 += x_end * x_end
mean_emp = s1 / n_paths
var_emp = s2 / n_paths - mean_emp ** 2
var_th = sigma_val ** 2 / (2.0 * theta)
print('empirical  E[X_T]   = {:+.4f}   (theory 0)'.format(mean_emp))
print('empirical  Var(X_T) = {:.4f}'.format(var_emp))
print('theory     sigma^2/(2 theta) = {:.4f}'.format(var_th))
print('relative error = {:.2f}%'.format(100 * abs(var_emp - var_th) / var_th))


## 4. Connection to diffusion models
The **variance-preserving SDE** of Song et al. (ICLR 2021),
$$dX_t = -\tfrac{1}{2}\beta(t)\,X_t\,dt + \sqrt{\beta(t)}\,dW_t,$$
is exactly an OU with time-varying rate $\theta(t) = \tfrac{1}{2}\beta(t)$
and matching noise $\sigma(t) = \sqrt{\beta(t)}$ chosen so that the stationary
law is $\mathcal N(0, 1)$. Score-based sampling reverses this SDE using a learned
$\nabla_x \log p_t(x)$. The *probability-flow ODE* used by ELF (arXiv:2605.10938)
is its deterministic limit, obtained by zeroing the diffusion while preserving
marginals.

## 5. Exercises
1. Replace the OU drift with the **double-well** $b(x) = x - x^3$ and look at
   the empirical distribution at large $T$ for $\sigma = 0.3$ vs $\sigma = 1.0$.
   How does noise change metastability?
2. Implement the **Milstein** correction
   $+\tfrac{1}{2}\sigma\,\partial_x\sigma\,((\Delta W_n)^2 - \Delta t)$
   and verify strong order $1$ on the GBM SDE (where the exact solution is known).
3. Verify Itô's formula numerically: for $Y_t = X_t^2$ with $X_t$ Brownian,
   compare empirical $\mathbb E[Y_t]$ to the prediction $t$ (i.e. $\mathrm{Var}(W_t)$).